In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

from huggingface_hub import list_repo_files, hf_hub_download

REPO_ID = "witgaw/METR-LA"
REPO_TYPE = "dataset"

DENSITY_THRESHOLD = 1000.0   # initially interpreted after inspecting units

In [ ]:
repo_files = list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

required_files = [
    "adj_mx.npy",
    "adj_mx_mapping.json",
    "distances.csv",
    "sensor_locations.csv"
]

file_paths = {}

for basename in required_files:

    matches = [
        f for f in repo_files
        if os.path.basename(f) == basename
    ]

    print(f"{basename}: {matches}")

    assert len(matches) == 1, (
        f"Expected exactly one {basename}; "
        f"found {matches}"
    )

    file_paths[basename] = matches[0]

print("\nResolved paths:")

for name, path in file_paths.items():
    print(f"{name:25s} -> {path}")

In [ ]:
local_files = {}

for name, repo_path in file_paths.items():

    local_files[name] = hf_hub_download(
        repo_id=REPO_ID,
        filename=repo_path,
        repo_type=REPO_TYPE
    )

    assert os.path.isfile(local_files[name])

print("All four files downloaded successfully.")

In [ ]:
adj_mx = np.load(
    local_files["adj_mx.npy"],
    allow_pickle=False
)

print("adj_mx.npy")
print("=" * 60)

print("Type  :", type(adj_mx))
print("Shape :", adj_mx.shape)
print("dtype :", adj_mx.dtype)

print("\nFirst 5 × 5 block:")
print(adj_mx[:5, :5])

assert isinstance(adj_mx, np.ndarray)
assert adj_mx.ndim == 2
assert adj_mx.shape[0] == adj_mx.shape[1]

N = adj_mx.shape[0]

print("\nNumber of graph nodes:", N)

In [ ]:
with open(
    local_files["adj_mx_mapping.json"],
    "r",
    encoding="utf-8"
) as f:
    mapping = json.load(f)

print("adj_mx_mapping.json")
print("=" * 60)

print("Python type:", type(mapping).__name__)

if isinstance(mapping, dict):

    print("\nKeys:")
    print(list(mapping.keys()))

    print("\nFirst few entries:")

    for key, value in list(mapping.items())[:5]:

        if isinstance(value, list) and len(value) > 10:
            print(
                f"{key}: {value[:10]} "
                f"... ({len(value)} entries)"
            )
        else:
            print(f"{key}: {value}")

elif isinstance(mapping, list):

    print("\nFirst 10 entries:")
    print(mapping[:10])

else:

    raise AssertionError(
        f"Unexpected mapping type: {type(mapping)}"
    )

In [ ]:
distances = pd.read_csv(
    local_files["distances.csv"]
)

print("distances.csv")
print("=" * 60)

print("Shape:", distances.shape)

print("\nColumn names:")
print(distances.columns.tolist())

print("\nFirst 5 rows:")
display(distances.head())

assert not distances.empty
assert len(distances.columns) > 0

In [ ]:
locations = pd.read_csv(
    local_files["sensor_locations.csv"]
)

print("sensor_locations.csv")
print("=" * 60)

print("Shape:", locations.shape)

print("\nColumn names:")
print(locations.columns.tolist())

print("\nFirst 5 rows:")
display(locations.head())

assert not locations.empty
assert len(locations.columns) > 0

In [ ]:
adj_no_self = adj_mx.copy()

np.fill_diagonal(
    adj_no_self,
    0
)

degrees = np.count_nonzero(
    adj_no_self,
    axis=1
)

degree_summary = pd.Series(
    degrees,
    name="degree"
).describe()

print("DEGREE DISTRIBUTION")
print("=" * 60)

print(degree_summary)

print("\nMinimum degree :", degrees.min())
print("Maximum degree :", degrees.max())
print("Mean degree    :", degrees.mean())
print("Median degree  :", np.median(degrees))

assert len(degrees) == N
assert (degrees >= 0).all()
assert (degrees <= N - 1).all()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    degrees,
    bins=np.arange(
        degrees.min(),
        degrees.max() + 2
    ) - 0.5
)

ax.set_title(
    "METR-LA Graph Degree Distribution"
)

ax.set_xlabel(
    "Number of Nonzero-Weight Neighbors"
)

ax.set_ylabel("Number of Sensors")

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------------
# Additional Network Topology Statistics
# -------------------------------------------------------------

import networkx as nx

G = nx.from_numpy_array(adj_no_self)

print("\n" + "=" * 60)
print("NETWORK TOPOLOGY SUMMARY")
print("=" * 60)

print(f"Number of nodes        : {G.number_of_nodes()}")
print(f"Number of edges        : {G.number_of_edges()}")

degrees = [d for _, d in G.degree()]

print(f"Average degree         : {np.mean(degrees):.2f}")
print(f"Median degree          : {np.median(degrees):.2f}")
print(f"Maximum degree         : {np.max(degrees)}")
print(f"Minimum degree         : {np.min(degrees)}")

print(f"Graph density          : {nx.density(G):.4f}")
print(f"Connected components   : {nx.number_connected_components(G)}")
print(f"Average clustering     : {nx.average_clustering(G):.4f}")

if nx.is_connected(G):
    print("\nPASS: Graph forms a single connected component.")
else:
    print("\nWARNING: Graph contains multiple connected components.")

In [ ]:
print("=" * 60)
print("DISTANCE FILE SCHEMA")
print("=" * 60)

print(f"Shape              : {distances.shape}")
print(f"Columns            : {list(distances.columns)}")

print("\nData Types")
print("-" * 60)
print(distances.dtypes)

print("\nMissing Values")
print("-" * 60)
print(distances.isnull().sum())

numeric_cols = distances.select_dtypes(include=np.number).columns.tolist()

print("\nNumeric Columns")
print("-" * 60)
print(numeric_cols)

print("\nPreview")
print("-" * 60)
display(distances.head())

In [ ]:
print("=" * 60)
print("DISTANCE STATISTICS")
print("=" * 60)

print(distances["cost"].describe())

print("\nDistance Quantiles (meters)")
print("-" * 60)

for q in [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    print(f"{int(q*100):>2}% : {distances['cost'].quantile(q):8.2f}")

print("\nDistance Range")
print("-" * 60)

print(f"Minimum distance : {distances['cost'].min():.2f} m")
print(f"Maximum distance : {distances['cost'].max():.2f} m")
print(f"Mean distance    : {distances['cost'].mean():.2f} m")
print(f"Median distance  : {distances['cost'].median():.2f} m")

In [ ]:
# -------------------------------------------------------------
# Pairwise Distance Validation
# -------------------------------------------------------------

SOURCE_COL = "from"
TARGET_COL = "to"
DIST_COL = "cost"

required_columns = [SOURCE_COL, TARGET_COL, DIST_COL]

missing = [
    col for col in required_columns
    if col not in distances.columns
]

assert len(missing) == 0, (
    f"Missing columns: {missing}"
)

pairwise = distances[
    required_columns
].copy()

positive_distances = pairwise[
    pairwise[DIST_COL] > 0
]

print("=" * 60)
print("PAIRWISE DISTANCE VALIDATION")
print("=" * 60)

print(f"Total distance records      : {len(pairwise):,}")
print(f"Positive distance records   : {len(positive_distances):,}")
print(f"Zero-distance records       : {(pairwise[DIST_COL] == 0).sum():,}")

print("\nDistance Summary (Positive Only)")
print("-" * 60)

print(
    positive_distances[DIST_COL].describe()
)

assert len(positive_distances) > 0

print("\nPASS: Pairwise distance table validated.")

In [ ]:
# -------------------------------------------------------------
# Pairwise Distance Distribution
# -------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    positive_distances[DIST_COL],
    bins=50,
    edgecolor="black"
)

ax.set_title("Distribution of Pairwise Sensor Distances")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("=" * 60)
print("PAIRWISE DISTANCE DISTRIBUTION")
print("=" * 60)

print(
    "The distribution summarizes the spatial separation "
    "between connected sensor pairs in the road network."
)

print(
    "Most connections occur within the middle distance "
    "range, while comparatively few sensor pairs are "
    "separated by very large distances."
)

In [ ]:
# -------------------------------------------------------------
# Neighborhood Radius Sensitivity Analysis
# -------------------------------------------------------------

thresholds = [500, 750, 1000, 1250, 1500, 2000]

print("=" * 60)
print("NEIGHBORHOOD RADIUS SENSITIVITY ANALYSIS")
print("=" * 60)

results = []

for radius in thresholds:

    neighbors = (
        pairwise.groupby(SOURCE_COL)[DIST_COL]
        .apply(lambda x: (x <= radius).sum())
    )

    results.append({
        "Radius (m)": radius,
        "Average Neighbors": round(neighbors.mean(), 2),
        "Median Neighbors": round(neighbors.median(), 2),
        "Minimum Neighbors": int(neighbors.min()),
        "Maximum Neighbors": int(neighbors.max()),
        "Std. Dev.": round(neighbors.std(), 2)
    })

results_df = pd.DataFrame(results)

display(results_df)

selected_radius = 1000

print("\nSelected Neighborhood Radius :", selected_radius, "m")

print(
    "\nA radius of 1000 m provides a balanced compromise "
    "between preserving local spatial relationships and "
    "maintaining sufficient neighborhood connectivity."
)

In [ ]:
# -------------------------------------------------------------
# Neighborhood Threshold Diagnostic
# -------------------------------------------------------------

threshold = selected_radius

within_threshold = (
    pairwise[DIST_COL] <= threshold
).sum()

outside_threshold = (
    pairwise[DIST_COL] > threshold
).sum()

total = len(pairwise)

print("=" * 60)
print("NEIGHBORHOOD THRESHOLD DIAGNOSTIC")
print("=" * 60)

print(f"Threshold                 : {threshold} m")
print(f"Pairs within threshold    : {within_threshold:,}")
print(f"Pairs outside threshold   : {outside_threshold:,}")
print(f"Percentage retained       : {within_threshold / total * 100:.2f}%")
print(f"Percentage excluded       : {outside_threshold / total * 100:.2f}%")

if within_threshold > 0:
    print("\nPASS: Threshold produces valid local neighborhoods.")
else:
    print("\nWARNING: No neighbors found within threshold.")

In [ ]:
# -------------------------------------------------------------
# Sensor Density Computation
# -------------------------------------------------------------

sensor_density = (
    pairwise[pairwise[DIST_COL] <= selected_radius]
    .groupby(SOURCE_COL)
    .size()
    .rename("neighbor_count")
    .reset_index()
)

print("=" * 60)
print("SENSOR DENSITY COMPUTATION")
print("=" * 60)

print(f"Neighborhood radius : {selected_radius} m")
print(f"Sensors analyzed    : {len(sensor_density)}")

display(sensor_density.head())

In [ ]:
# -------------------------------------------------------------
# Sensor Density Statistics
# -------------------------------------------------------------

print("=" * 60)
print("SENSOR DENSITY STATISTICS")
print("=" * 60)

print(sensor_density["neighbor_count"].describe())

print("\nDensity Quantiles")
print("-" * 60)

for q in [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    print(
        f"{int(q*100):>2}% : "
        f"{sensor_density['neighbor_count'].quantile(q):.0f}"
    )

print("\nAdditional Statistics")
print("-" * 60)

print(f"Mean neighbors   : {sensor_density['neighbor_count'].mean():.2f}")
print(f"Median neighbors : {sensor_density['neighbor_count'].median():.0f}")
print(f"Std. deviation   : {sensor_density['neighbor_count'].std():.2f}")
print(f"Maximum neighbors: {sensor_density['neighbor_count'].max()}")
print(f"Minimum neighbors: {sensor_density['neighbor_count'].min()}")

In [ ]:
# -------------------------------------------------------------
# Sensor Location Dataset Inspection
# -------------------------------------------------------------

print("=" * 60)
print("SENSOR LOCATION DATASET")
print("=" * 60)

print(f"Shape      : {locations.shape}")
print(f"Columns    : {list(locations.columns)}")

print("\nData Types")
print("-" * 60)
print(locations.dtypes)

print("\nMissing Values")
print("-" * 60)
print(locations.isnull().sum())

print("\nPreview")
print("-" * 60)
display(locations.head())

In [ ]:
# -------------------------------------------------------------
# Merge Density with Sensor Locations
# -------------------------------------------------------------

density_map = sensor_density.merge(
    locations,
    left_on="from",
    right_on="sensor_id",
    how="left"
)

print("=" * 60)
print("DENSITY-LOCATION MERGE")
print("=" * 60)

print(f"Total sensors in density table : {len(sensor_density)}")
print(f"Merged records                 : {len(density_map)}")
print(f"Matched locations              : {density_map['latitude'].notna().sum()}")

print("\nMissing Coordinates")
print("-" * 60)

print(density_map[["latitude", "longitude"]].isnull().sum())

display(density_map.head())

In [ ]:
# -------------------------------------------------------------
# Geographic Distribution of Sensor Density
# -------------------------------------------------------------

plt.figure(figsize=(8, 8))

plt.scatter(
    density_map["longitude"],
    density_map["latitude"],
    c=density_map["neighbor_count"],
    cmap="viridis",
    s=40,
    alpha=0.8
)

plt.colorbar(label="Neighbor Count")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Spatial Distribution of Sensor Density ({selected_radius} m)")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------------
# Final Validation Summary
# -------------------------------------------------------------

print("=" * 60)
print("FINAL VALIDATION SUMMARY")
print("=" * 60)

print(f"Road network nodes          : {G.number_of_nodes()}")
print(f"Road network edges          : {G.number_of_edges()}")
print(f"Connected components        : {nx.number_connected_components(G)}")

print(f"\nDistance records           : {len(pairwise):,}")
print(f"Neighborhood radius        : {selected_radius} m")

print(f"\nSensors with coordinates   : {len(locations)}")
print(f"Sensors with density score : {len(sensor_density)}")
print(f"Merged sensor records      : {len(density_map)}")

print("\nPASS: Graph topology validated.")
print("PASS: Distance dataset validated.")
print("PASS: Neighborhood density computed.")
print("PASS: Geographic coordinates successfully merged.")
print("\nNotebook 05 completed successfully.")

In [ ]:
# -------------------------------------------------------------
# Haversine Distance Validation
# -------------------------------------------------------------

from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance between two geographic coordinates.
    Returns distance in meters.
    """
    R = 6371000  # Earth radius (m)

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


# -------------------------------------------------------------
# Merge geographic coordinates
# -------------------------------------------------------------

hv = pairwise.merge(
    locations,
    left_on="from",
    right_on="sensor_id",
    how="left"
).rename(columns={
    "latitude": "lat_from",
    "longitude": "lon_from"
})

hv = hv.merge(
    locations,
    left_on="to",
    right_on="sensor_id",
    how="left",
    suffixes=("", "_to")
).rename(columns={
    "latitude": "lat_to",
    "longitude": "lon_to"
})


# -------------------------------------------------------------
# Compute Haversine distances
# -------------------------------------------------------------

hv["haversine_m"] = hv.apply(
    lambda r: haversine(
        r["lat_from"],
        r["lon_from"],
        r["lat_to"],
        r["lon_to"]
    )
    if (
        pd.notnull(r["lat_from"])
        and pd.notnull(r["lat_to"])
    )
    else np.nan,
    axis=1
)

valid_hv = hv.dropna(subset=["haversine_m"]).copy()

valid_hv["road_haversine_ratio"] = (
    valid_hv["cost"] /
    valid_hv["haversine_m"].replace(0, np.nan)
)

ratio = valid_hv["road_haversine_ratio"].dropna()

coverage = len(valid_hv) / len(hv) * 100

greater_pct = (
    valid_hv["cost"] >= valid_hv["haversine_m"]
).mean() * 100


# -------------------------------------------------------------
# Validation Summary
# -------------------------------------------------------------

print("=" * 60)
print("HAVERSINE DISTANCE VALIDATION")
print("=" * 60)

print(f"Total road-network pairs        : {len(hv):,}")
print(f"Pairs with valid coordinates    : {len(valid_hv):,}")
print(f"Coordinate coverage             : {coverage:.2f}%")

print("\nRoad-network Distance Statistics")
print("-" * 60)
print(valid_hv["cost"].describe())

print("\nHaversine Distance Statistics")
print("-" * 60)
print(valid_hv["haversine_m"].describe())

print("\nRoad / Haversine Distance Ratio")
print("-" * 60)
print(ratio.describe())

print("\nRobust Ratio Summary")
print("-" * 60)
print(f"Median Ratio                    : {ratio.median():.2f}")
print(f"IQR                             : {ratio.quantile(0.25):.2f} - {ratio.quantile(0.75):.2f}")

print("\nValidation Checks")
print("-" * 60)
print(f"Road distance ≥ Haversine       : {greater_pct:.2f}% of valid pairs")

if greater_pct >= 95:
    print("PASS: Geographic consistency successfully validated.")
else:
    print("WARNING: Unexpected road-network distance relationships detected.")

print("\nInterpretation")
print("-" * 60)

print(
    f"Among {len(hv):,} road-network distance pairs, "
    f"{len(valid_hv):,} ({coverage:.2f}%) were directly matched with valid "
    "geographic coordinates and used for validation. This subset represents "
    "the sensor pairs for which both endpoints are available in the location "
    "dataset. The median road-to-Haversine distance ratio of "
    f"{ratio.median():.2f} indicates that travel along the road network is "
    "typically longer than the corresponding straight-line geographic distance. "
    f"Furthermore, {greater_pct:.2f}% of validated pairs satisfy the expected "
    "relationship that road-network distance is greater than or equal to "
    "Haversine distance, confirming the geographic consistency of the road "
    "network graph. The few extreme ratio values are attributable to sensor "
    "pairs with very small straight-line distances, where even modest road "
    "distances produce large ratios."
)